<a href="https://colab.research.google.com/github/Aswathi281099/Generative-Artificial-Intelligence/blob/main/AI_TASK_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TASK 4**

In [1]:
import numpy as np
import scipy.spatial.distance as dist
import networkx as nx
import heapq
from typing import List, Tuple, Set, Dict

In [4]:
import numpy as np
import scipy.spatial.distance as dist
import networkx as nx
import heapq
from typing import List, Tuple, Set, Dict


class HNSWIndexFromScratch:
    """
    In-Memory Hierarchical Navigable Small World (HNSW) Vector Index.
    Built using NumPy, SciPy, and NetworkX.
    """
    def __init__(self, dim: int, M: int = 16, ef_construction: int = 64, ef_search: int = 32):
        self.dim = dim
        self.M = M
        self.M_max0 = 2 * M  # Max connections for layer 0
        self.ef_construction = ef_construction
        self.ef_search = ef_search
        self.mL = 1.0 / np.log(M)

        self.vectors: Dict[int, np.ndarray] = {}
        self.entry_point: int = None
        self.max_level: int = -1

        self.layer_graphs: List[nx.Graph] = []

    def _cosine_distance(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """Computes Cosine Distance between two vectors."""
        d = dist.cosine(vec1, vec2)
        return float(d) if not np.isnan(d) else 0.0

    def _get_random_level(self) -> int:
        """Probability-driven layer assignment using exponential distribution."""
        unif = np.random.uniform(1e-6, 1.0)
        return int(np.floor(-np.log(unif) * self.mL))

    def _search_layer(
        self, q: np.ndarray, entry_points: List[int], ef: int, layer_idx: int
    ) -> List[Tuple[float, int]]:
        """Greedy search inside a specific graph layer returning ef nearest neighbors."""
        graph = self.layer_graphs[layer_idx]

        # Guard: Filter out entry points not present in this specific layer
        valid_eps = [ep for ep in entry_points if graph.has_node(ep)]
        if not valid_eps:
            # Fallback to any node in graph if entry points are missing
            valid_eps = list(graph.nodes())[:1] if graph.number_of_nodes() > 0 else []

        if not valid_eps:
            return []

        visited: Set[int] = set(valid_eps)
        candidates = []
        W = []

        for ep in valid_eps:
            d = self._cosine_distance(q, self.vectors[ep])
            heapq.heappush(candidates, (d, ep))
            heapq.heappush(W, (-d, ep))

        while candidates:
            c_dist, c_node = heapq.heappop(candidates)
            f_dist = -W[0][0]

            if c_dist > f_dist:
                break

            for neighbor in graph.neighbors(c_node):
                if neighbor not in visited:
                    visited.add(neighbor)
                    f_dist = -W[0][0]
                    d_neighbor = self._cosine_distance(q, self.vectors[neighbor])

                    if d_neighbor < f_dist or len(W) < ef:
                        heapq.heappush(candidates, (d_neighbor, neighbor))
                        heapq.heappush(W, (-d_neighbor, neighbor))
                        if len(W) > ef:
                            heapq.heappop(W)

        return sorted([(-d, node) for d, node in W], key=lambda x: x[0])

    def insert(self, node_id: int, vector: np.ndarray):
        """Inserts a new vector into the multi-layer graph index."""
        vector = np.asarray(vector, dtype=np.float32)
        self.vectors[node_id] = vector
        l = self._get_random_level()

        # Ensure graph levels exist
        while len(self.layer_graphs) <= max(l, self.max_level, 0):
            self.layer_graphs.append(nx.Graph())

        # First element initialization
        if self.entry_point is None:
            self.entry_point = node_id
            self.max_level = l
            for level in range(l + 1):
                self.layer_graphs[level].add_node(node_id)
            return

        curr_ep = [self.entry_point]
        L = self.max_level

        # Phase 1: Skip top layers down to l + 1
        for level in range(L, l, -1):
            res = self._search_layer(vector, curr_ep, ef=1, layer_idx=level)
            if res:
                curr_ep = [res[0][1]]

        # Phase 2: Connect inside layers from min(L, l) down to 0
        for level in range(min(L, l), -1, -1):
            self.layer_graphs[level].add_node(node_id)
            neighbors_res = self._search_layer(vector, curr_ep, ef=self.ef_construction, layer_idx=level)

            max_m = self.M_max0 if level == 0 else self.M
            selected_neighbors = neighbors_res[:max_m]

            # Add bidirectional edges
            for dist_val, nbr in selected_neighbors:
                self.layer_graphs[level].add_edge(node_id, nbr, weight=dist_val)

            # Prune over-connected neighbors
            for dist_val, nbr in selected_neighbors:
                nbr_edges = list(self.layer_graphs[level].edges(nbr, data=True))
                if len(nbr_edges) > max_m:
                    sorted_edges = sorted(nbr_edges, key=lambda e: e[2].get('weight', 0.0))
                    for u, v, _ in sorted_edges[max_m:]:
                        if self.layer_graphs[level].has_edge(u, v):
                            self.layer_graphs[level].remove_edge(u, v)

            if neighbors_res:
                curr_ep = [res[1] for res in neighbors_res]

        # Update global entry point if new node has a higher level
        if l > self.max_level:
            self.max_level = l
            self.entry_point = node_id

    def search(self, query: np.ndarray, k: int = 5) -> List[Tuple[int, float, float]]:
        """Executes hierarchical vector search and outputs cosine similarity rankings."""
        if self.entry_point is None:
            return []

        query = np.asarray(query, dtype=np.float32)
        curr_ep = [self.entry_point]

        # Traversal down to level 1
        for level in range(self.max_level, 0, -1):
            res = self._search_layer(query, curr_ep, ef=1, layer_idx=level)
            if res:
                curr_ep = [res[0][1]]

        # Search level 0
        results = self._search_layer(query, curr_ep, ef=max(self.ef_search, k), layer_idx=0)

        top_k = results[:k]
        return [(node_id, dist_val, 1.0 - dist_val) for dist_val, node_id in top_k]

In [5]:
if __name__ == "__main__":
    np.random.seed(42)
    dim = 128
    num_vectors = 500

    print("1. Initializing Custom HNSW Index...")
    hnsw = HNSWIndexFromScratch(dim=dim, M=16, ef_construction=64, ef_search=32)

    print("2. Generating & Indexing Random Embeddings...")
    dataset = np.random.randn(num_vectors, dim).astype(np.float32)
    for i in range(num_vectors):
        hnsw.insert(node_id=i, vector=dataset[i])

    print(f"\nIndex Construction Complete!")
    print(f"Total Constructed Layers: {len(hnsw.layer_graphs)}")
    for idx, g in enumerate(hnsw.layer_graphs):
        print(f" - Layer {idx}: {g.number_of_nodes()} Nodes, {g.number_of_edges()} Edges")

    print("\n3. Querying Top-5 Nearest Neighbors...")
    query_vec = np.random.randn(dim).astype(np.float32)
    top_k_results = hnsw.search(query=query_vec, k=5)

    print("\n--- Cosine Similarity Search Results ---")
    print(f"{'Rank':<5} | {'Node ID':<10} | {'Cosine Distance':<18} | {'Cosine Similarity':<18}")
    print("-" * 60)
    for rank, (node_id, distance, similarity) in enumerate(top_k_results, start=1):
        print(f"{rank:<5} | {node_id:<10} | {distance:<18.4f} | {similarity:<18.4f}")

1. Initializing Custom HNSW Index...
2. Generating & Indexing Random Embeddings...

Index Construction Complete!
Total Constructed Layers: 3
 - Layer 0: 500 Nodes, 7623 Edges
 - Layer 1: 28 Nodes, 212 Edges
 - Layer 2: 0 Nodes, 0 Edges

3. Querying Top-5 Nearest Neighbors...

--- Cosine Similarity Search Results ---
Rank  | Node ID    | Cosine Distance    | Cosine Similarity 
------------------------------------------------------------
1     | 158        | 0.7433             | 0.2567            
2     | 389        | 0.7747             | 0.2253            
3     | 86         | 0.7791             | 0.2209            
4     | 149        | 0.7879             | 0.2121            
5     | 287        | 0.7936             | 0.2064            
